In [1]:
import os

# Windows Hadoop configuration
os.environ["HADOOP_HOME"] = r"D:\NOPIS\Phase_2\winutils"
os.environ["PATH"] = (
    r"D:\NOPIS\Phase_2\winutils\bin;"
    + os.environ["PATH"]
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NOPIS_SP5_Performance")
    .master("local[*]")
    .config("spark.hadoop.home.dir", r"D:\NOPIS\Phase_2\winutils")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

c:\Users\nalin.karthik\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
HADOOP_HOME: D:\NOPIS\Phase_2\winutils


In [2]:
# Load the existing SP3 hourly grid summary

hourly_df = spark.read.parquet(
    r"D:\NOPIS\Phase_2\outputs\hourly_grid_summary"
)

hourly_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- grid_id: integer (nullable = true)
 |-- sms_in: double (nullable = true)
 |-- sms_out: double (nullable = true)
 |-- call_in: double (nullable = true)
 |-- call_out: double (nullable = true)
 |-- internet_activity: double (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- total_sms: double (nullable = true)
 |-- total_calls: double (nullable = true)
 |-- total_activity: double (nullable = true)
 |-- internet_share: double (nullable = true)



In [3]:
# 56. Run explain() on a hotspot aggregation and read the physical plan.

from pyspark.sql import functions as F

hotspot_agg_df = (
    hourly_df
    .groupBy("grid_id")
    .agg(
        F.sum("total_activity").alias("total_activity")
    )
)

hotspot_agg_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[grid_id#1], functions=[sum(total_activity#11)])
   +- Exchange hashpartitioning(grid_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=11]
      +- HashAggregate(keys=[grid_id#1], functions=[partial_sum(total_activity#11)])
         +- FileScan parquet [grid_id#1,total_activity#11] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/D:/NOPIS/Phase_2/outputs/hourly_grid_summary], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<grid_id:int,total_activity:double>




In [4]:
# 57. Cache a reused cleaned DataFrame and compare repeated action timings.

cleaned_df = hourly_df.select(
    "timestamp",
    "grid_id",
    "total_activity"
)

print("Cleaned DataFrame created.")

Cleaned DataFrame created.


In [5]:
import time

def measure_count(df):
    start = time.perf_counter()
    count = df.count()
    end = time.perf_counter()
    
    return count, end - start

In [6]:
print("BEFORE CACHING")

count_1, time_1 = measure_count(cleaned_df)
print(f"First count : {count_1:,} rows | {time_1:.3f} seconds")

count_2, time_2 = measure_count(cleaned_df)
print(f"Second count: {count_2:,} rows | {time_2:.3f} seconds")

BEFORE CACHING
First count : 1,679,994 rows | 2.096 seconds
Second count: 1,679,994 rows | 0.353 seconds


In [8]:
cached_df = cleaned_df.cache()

print("DataFrame cached.")

DataFrame cached.


In [9]:
print("AFTER CACHING")

count_3, time_3 = measure_count(cached_df)
print(f"First cached count : {count_3:,} rows | {time_3:.3f} seconds")

count_4, time_4 = measure_count(cached_df)
print(f"Second cached count: {count_4:,} rows | {time_4:.3f} seconds")

AFTER CACHING
First cached count : 1,679,994 rows | 1.385 seconds
Second cached count: 1,679,994 rows | 0.265 seconds


In [10]:
cached_df.unpersist()

print("Cache released.")

Cache released.


In [11]:
# 58. Repartition by date or another suitable key and observe the partition counts.

print("Current number of partitions:", hourly_df.rdd.getNumPartitions())

Current number of partitions: 15


In [12]:
repartitioned_df = hourly_df.repartition("date")

print(
    "Partitions after repartition by date:",
    repartitioned_df.rdd.getNumPartitions()
)

Partitions after repartition by date: 7


In [14]:
# 59. Demonstrate column pruning by selecting only the required fields before an aggregation.

pruned_df = hourly_df.select(
    "grid_id",
    "total_activity"
)

pruned_agg_df = (
    pruned_df
    .groupBy("grid_id")
    .agg(
        F.sum("total_activity").alias("total_activity")
    )
)

pruned_agg_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[grid_id#1], functions=[sum(total_activity#11)])
   +- Exchange hashpartitioning(grid_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=230]
      +- HashAggregate(keys=[grid_id#1], functions=[partial_sum(total_activity#11)])
         +- FileScan parquet [grid_id#1,total_activity#11] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/D:/NOPIS/Phase_2/outputs/hourly_grid_summary], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<grid_id:int,total_activity:double>




In [16]:
# 60. Broadcast the static grid lookup from SP4 and compare the plan against the standard join.

import json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

geojson_path = r"D:\NOPIS\data\milano-grid.geojson"

with open(geojson_path, "r") as f:
    geo_data = json.load(f)

grid_lookup_data = [
    (
        feature["properties"]["cellId"],
        json.dumps(feature["geometry"])
    )
    for feature in geo_data["features"]
]

grid_schema = StructType([
    StructField("grid_id", IntegerType(), nullable=False),
    StructField("geometry", StringType(), nullable=False)
])

grid_lookup_df = spark.createDataFrame(
    grid_lookup_data,
    schema=grid_schema
)

print("Grid lookup DataFrame created.")

Grid lookup DataFrame created.


In [17]:
# 60. Broadcast the static grid lookup from SP4 and compare the plan against the standard join.

standard_join_df = hourly_df.join(
    grid_lookup_df,
    on="grid_id",
    how="left"
)

print("STANDARD JOIN PLAN")
standard_join_df.explain()

STANDARD JOIN PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [grid_id#1, timestamp#0, sms_in#2, sms_out#3, call_in#4, call_out#5, internet_activity#6, date#7, hour#8, total_sms#9, total_calls#10, total_activity#11, internet_share#12, geometry#239]
   +- SortMergeJoin [grid_id#1], [grid_id#238], LeftOuter
      :- Sort [grid_id#1 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(grid_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=278]
      :     +- FileScan parquet [timestamp#0,grid_id#1,sms_in#2,sms_out#3,call_in#4,call_out#5,internet_activity#6,date#7,hour#8,total_sms#9,total_calls#10,total_activity#11,internet_share#12] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/D:/NOPIS/Phase_2/outputs/hourly_grid_summary], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<timestamp:timestamp,grid_id:int,sms_in:double,sms_out:double,call_in:double,call_out:doubl...
      +- Sort [grid_id#238 ASC NULLS FIRS

In [18]:
from pyspark.sql.functions import broadcast

broadcast_join_df = hourly_df.join(
    broadcast(grid_lookup_df),
    on="grid_id",
    how="left"
)

print("BROADCAST JOIN PLAN")
broadcast_join_df.explain()

BROADCAST JOIN PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [grid_id#1, timestamp#0, sms_in#2, sms_out#3, call_in#4, call_out#5, internet_activity#6, date#7, hour#8, total_sms#9, total_calls#10, total_activity#11, internet_share#12, geometry#239]
   +- BroadcastHashJoin [grid_id#1], [grid_id#238], LeftOuter, BuildRight, false, false
      :- FileScan parquet [timestamp#0,grid_id#1,sms_in#2,sms_out#3,call_in#4,call_out#5,internet_activity#6,date#7,hour#8,total_sms#9,total_calls#10,total_activity#11,internet_share#12] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/D:/NOPIS/Phase_2/outputs/hourly_grid_summary], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<timestamp:timestamp,grid_id:int,sms_in:double,sms_out:double,call_in:double,call_out:doubl...
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=300]
         +- Scan ExistingRDD[grid_id#2

61. Discuss why over-partitioning a small local dataset makes performance worse.

Observation: Over-partitioning a small local dataset can increase task scheduling and shuffle overhead without providing meaningful additional parallelism. Therefore, more partitions are not automatically better; partitioning should match the workload and available resources.

62. Document three performance observations with the evidence that supports each.

Observation 1 — Aggregation causes a shuffle

Evidence from Q56:

Exchange hashpartitioning(grid_id#1, 200)

The physical plan showed:

FileScan
   ↓
Partial HashAggregate
   ↓
Exchange
   ↓
Final HashAggregate

Document:

Observation 1: The hotspot aggregation on grid_id requires a shuffle. The physical plan contains Exchange hashpartitioning(grid_id, 200), showing that Spark redistributes data by grid_id before the final aggregation.

Observation 2 — Caching helped repeated use

Evidence from Q57:

Before cache — second count: 0.353 seconds
After cache  — second count: 0.265 seconds

That's approximately a 25% reduction in the measured time.

Document:

Observation 2: Caching the reused DataFrame improved the repeated count() action. The second action decreased from 0.353 seconds without caching to 0.265 seconds with caching. The improvement was modest on this local training dataset.

Notice that we're saying modest improvement, not "caching always makes Spark faster."

Observation 3 — Spark automatically pruned columns

Evidence from Q56/Q59:

FileScan parquet [grid_id#1,total_activity#11]

Our source DataFrame has 13 columns, but Spark's physical plan reads only:

grid_id
total_activity

Even after explicitly selecting those columns in Q59, the physical plan remained essentially the same.

Document:

Observation 3: Spark automatically performed column pruning for the aggregation. The physical plan's FileScan parquet reads only grid_id and total_activity, even though the source contains additional columns. Explicitly selecting these columns did not materially change the physical plan.

SP6

In [19]:
# 63. Write the clean activity data as Parquet.

print("Rows:", hourly_df.count())
hourly_df.printSchema()

Rows: 1679994
root
 |-- timestamp: timestamp (nullable = true)
 |-- grid_id: integer (nullable = true)
 |-- sms_in: double (nullable = true)
 |-- sms_out: double (nullable = true)
 |-- call_in: double (nullable = true)
 |-- call_out: double (nullable = true)
 |-- internet_activity: double (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- total_sms: double (nullable = true)
 |-- total_calls: double (nullable = true)
 |-- total_activity: double (nullable = true)
 |-- internet_share: double (nullable = true)



In [20]:
output_path = r"D:\NOPIS\data\processed\activity"

hourly_df.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Clean activity data written to:")
print(output_path)

Clean activity data written to:
D:\NOPIS\data\processed\activity


In [21]:
import os

print(os.listdir(output_path))

['.part-00000-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00001-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00002-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00003-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00004-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00005-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00006-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00007-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00008-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00009-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00010-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00011-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-00012-bb27fd58-8d62-4926-98dd-1b4f7f6cd01e-c000.snappy.parquet.crc', '.part-0001

In [22]:
# 64. Partition the cleaned output by date.

partitioned_output_path = r"D:\NOPIS\data\processed\activity"

hourly_df.write \
    .mode("overwrite") \
    .partitionBy("date") \
    .parquet(partitioned_output_path)

print("Cleaned activity data written partitioned by date.")

Cleaned activity data written partitioned by date.


In [23]:
import os

folders = [
    item for item in os.listdir(partitioned_output_path)
    if item.startswith("date=")
]

print("Date partitions:")
for folder in sorted(folders):
    print(folder)

Date partitions:
date=2013-11-01
date=2013-11-02
date=2013-11-03
date=2013-11-04
date=2013-11-05
date=2013-11-06
date=2013-11-07


In [24]:
# 65. Write hourly_grid_summary as Parquet at one record per grid and hour. Keep full Polygon geometry in the static grid reference rather than duplicating it into every analytics record.

hourly_grid_summary = (
    hourly_df
    .groupBy("grid_id", "timestamp", "date", "hour")
    .agg(
        F.sum("total_sms").alias("total_sms"),
        F.sum("total_calls").alias("total_calls"),
        F.sum("total_activity").alias("total_activity")
    )
)

hourly_grid_summary.printSchema()

root
 |-- grid_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- total_sms: double (nullable = true)
 |-- total_calls: double (nullable = true)
 |-- total_activity: double (nullable = true)



In [25]:
duplicate_check = (
    hourly_grid_summary
    .groupBy("grid_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate grid_id + timestamp rows:", duplicate_check.count())

Duplicate grid_id + timestamp rows: 0


In [26]:
analytics_output_path = r"D:\NOPIS\data\analytics\hourly_grid_summary"

hourly_grid_summary.write \
    .mode("overwrite") \
    .parquet(analytics_output_path)

print("Hourly grid summary written to:")
print(analytics_output_path)

Hourly grid summary written to:
D:\NOPIS\data\analytics\hourly_grid_summary


In [27]:
# 66. Write a small dashboard summary as CSV for easy inspection, and retain milano-grid.geojson separately under data/reference/ for map rendering.

dashboard_summary = (
    hourly_grid_summary
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("grid_id").alias("unique_grids"),
        F.sum("total_sms").alias("total_sms"),
        F.sum("total_calls").alias("total_calls"),
        F.sum("total_activity").alias("total_activity")
    )
)

dashboard_summary.show()

+-------------+------------+-------------------+-------------------+-------------------+
|total_records|unique_grids|          total_sms|        total_calls|     total_activity|
+-------------+------------+-------------------+-------------------+-------------------+
|      1679994|       10000|7.120683090059994E7|6.482891661920006E7|8.236518001292005E8|
+-------------+------------+-------------------+-------------------+-------------------+



In [28]:
dashboard_output_path = r"D:\NOPIS\Phase_2\outputs\dashboard_summary"

dashboard_summary.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(dashboard_output_path)

print("Dashboard summary written to:")
print(dashboard_output_path)

Dashboard summary written to:
D:\NOPIS\Phase_2\outputs\dashboard_summary


In [30]:
reference_geojson = r"D:\NOPIS\data\milano-grid.geojson"

print("GeoJSON exists:", os.path.exists(reference_geojson))
print("Reference file:", reference_geojson)

GeoJSON exists: True
Reference file: D:\NOPIS\data\milano-grid.geojson


In [31]:
# 67. Read the Parquet output back and validate schema and counts.

roundtrip_df = spark.read.parquet(
    r"D:\NOPIS\data\analytics\hourly_grid_summary"
)

print("Round-trip row count:", roundtrip_df.count())

print("\nRound-trip schema:")
roundtrip_df.printSchema()

Round-trip row count: 1679994

Round-trip schema:
root
 |-- grid_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- total_sms: double (nullable = true)
 |-- total_calls: double (nullable = true)
 |-- total_activity: double (nullable = true)



In [32]:
duplicate_roundtrip = (
    roundtrip_df
    .groupBy("grid_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate (grid_id, timestamp) rows after round trip:",
    duplicate_roundtrip.count()
)

Duplicate (grid_id, timestamp) rows after round trip: 0


In [33]:
# 68. Compare file sizes and explain the benefits of columnar storage.

csv_comparison_path = r"D:\NOPIS\Phase_2\outputs\hourly_grid_summary_csv"

hourly_grid_summary.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(csv_comparison_path)

print("CSV comparison dataset written.")

CSV comparison dataset written.


In [34]:
import os

def get_directory_size(path):
    total_size = 0
    
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)
    
    return total_size

parquet_size = get_directory_size(
    r"D:\NOPIS\data\analytics\hourly_grid_summary"
)

csv_size = get_directory_size(
    csv_comparison_path
)

print(f"Parquet size: {parquet_size:,} bytes")
print(f"CSV size:     {csv_size:,} bytes")
print(f"Parquet size: {parquet_size / (1024**2):.2f} MB")
print(f"CSV size:     {csv_size / (1024**2):.2f} MB")

Parquet size: 40,829,340 bytes
CSV size:     142,716,880 bytes
Parquet size: 38.94 MB
CSV size:     136.11 MB


## 68. Compare file sizes and explain the benefits of columnar storage.

The same `hourly_grid_summary` dataset was written in both Parquet and CSV formats.

- Parquet size: 38.94 MB
- CSV size: 136.11 MB

Parquet used approximately 71% less storage than CSV.

Parquet is a columnar format, so Spark can read only the required columns for an
analytical query instead of processing the entire dataset. Parquet also supports
efficient compression, which helps reduce storage size and improves analytical
read performance.

CSV remains useful for small dashboard summaries because it is simple and easy
to inspect, while Parquet is more appropriate for Spark processed and analytics
data.

In [35]:
# 69. Create spark/telecom_pipeline.py with read_raw(), clean(), aggregate(), enrich(), write_outputs() and main().

import os

project_root = r"D:\NOPIS"

for root, dirs, files in os.walk(project_root):
    level = root.replace(project_root, "").count(os.sep)
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")
        for file in files:
            print("  " * (level + 1) + file)

NOPIS/
  .gitignore
  .git/
    COMMIT_EDITMSG
    config
    description
    HEAD
    index
    hooks/
      applypatch-msg.sample
      commit-msg.sample
      fsmonitor-watchman.sample
      post-update.sample
      pre-applypatch.sample
      pre-commit.sample
      pre-merge-commit.sample
      pre-push.sample
      pre-rebase.sample
      pre-receive.sample
      prepare-commit-msg.sample
      push-to-checkout.sample
      sendemail-validate.sample
      update.sample
    info/
      exclude
    logs/
      HEAD
    objects/
    refs/
  data/
    milano-grid.geojson
    sms-call-internet-mi-2013-11-01.csv
    sms-call-internet-mi-2013-11-02.csv
    sms-call-internet-mi-2013-11-03.csv
    sms-call-internet-mi-2013-11-04.csv
    sms-call-internet-mi-2013-11-05.csv
    sms-call-internet-mi-2013-11-06.csv
    sms-call-internet-mi-2013-11-07.csv
    analytics/
    data_Set/
      import.py
    processed/
  Phase_1/
    docs/
      np3_alert_limitations.md
    notebooks/
      np1.ipy